In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))   # rend mlproject importable depuis todo/

from mlproject.data import load_data, split
from mlproject.config import TARGET, NUMERIC_FEATURES, CATEGORICAL_FEATURES

df = load_data()
print("Lignes, colonnes :", df.shape)
print("Cible :", TARGET)
print("Répartition des classes :")
print(df[TARGET].value_counts())

Lignes, colonnes : (284807, 31)
Cible : Class
Répartition des classes :
Class
0    284315
1       492
Name: count, dtype: int64


In [2]:
from mlproject.train import build_model
from sklearn.metrics import f1_score, roc_auc_score
import pandas as pd

# Découpage train/test avec TON code (le même que train.py)
x_train, x_test, y_train, y_test = split(df)
print("Train :", x_train.shape, " Test :", x_test.shape)

# Petite fonction pour évaluer n'importe quel pipeline de la même façon
def evaluer(nom, model):
    model.fit(x_train, y_train)
    proba = model.predict_proba(x_test)[:, 1]
    preds = (proba >= 0.5).astype(int)
    f1 = f1_score(y_test, preds)
    auc = roc_auc_score(y_test, proba)
    print(f"{nom:25s}  f1={f1:.3f}  roc_auc={auc:.3f}")
    return {"modele": nom, "f1": f1, "roc_auc": auc}

resultats = []

# Référence = exactement le modèle de train.py (régression logistique)
resultats.append(evaluer("LogisticRegression (ref)", build_model()))

Train : (227845, 30)  Test : (56962, 30)
LogisticRegression (ref)   f1=0.717  roc_auc=0.961


In [3]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from mlproject.features import build_preprocessor
from sklearn.pipeline import Pipeline

# On réutilise TON préprocesseur (StandardScaler sur tes 30 colonnes),
# et on remplace juste le classifieur final.

# Random Forest — n_jobs=-1 pour utiliser tous les cœurs
rf = Pipeline(steps=[
    ("preprocessor", build_preprocessor()),
    ("clf", RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)),
])
resultats.append(evaluer("RandomForest", rf))

# XGBoost — scale_pos_weight gère le déséquilibre (≈ nb négatifs / nb positifs)
xgb = Pipeline(steps=[
    ("preprocessor", build_preprocessor()),
    ("clf", XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.1,
        scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
        eval_metric="logloss", random_state=42, n_jobs=-1,
    )),
])
resultats.append(evaluer("XGBoost", xgb))

# Tableau comparatif trié par f1
import pandas as pd
comparatif = pd.DataFrame(resultats).sort_values("f1", ascending=False).reset_index(drop=True)
comparatif

RandomForest               f1=0.876  roc_auc=0.957
XGBoost                    f1=0.869  roc_auc=0.969


,modele,f1,roc_auc
0,RandomForest,0.875676,0.957215
1,XGBoost,0.869110,0.969412
2,LogisticRegression (ref),0.716763,0.960549
